In [4]:
import sympy as sp
import numpy as np
from scipy.integrate import simpson  # Fixed: np.trapz -> simpson

print("=== SymPy: Full 4D Chern-Ricci via Randers (Pfeifer-Wohlfarth) ===")

# Symbole
t, r, theta, phi = sp.symbols('t r theta phi', positive=True)
y0, y1, y2, y3 = sp.symbols('y0 y1 y2 y3')
b0, eps = sp.symbols('b0 epsilon', positive=True)

# Randers F^2 (Pfeifer Sec 3.2)
alpha2 = -y0**2 + y1**2 + r**2 * y2**2 + (r*sp.sin(theta))**2 * y3**2
by = b0 * sp.cos(theta) * y1
F2 = alpha2 + by**2
F = sp.sqrt(F2)

# g_ij excerpt
g11 = sp.simplify(sp.diff(F2, y1, y1)/2)  # rr
g12 = sp.simplify(sp.diff(F2, y1, y2)/2)
print("g_rr:", sp.latex(g11))
print("g_r theta:", sp.latex(g12))

# Cartan Torsion
C_r_th_ph = -b0 * sp.sin(theta) * y2 * y3 / F**2
print("C^r_theta phi:", sp.latex(C_r_th_ph))

# Chern-Ricci approx
C2 = C_r_th_ph**2
RF_approx = - (3/4) * C2 / r**2 + b0**3 / r**2
print("R_F approx:", sp.latex(RF_approx.series(b0,0,4).removeO()))

# RG Flow
ell, bCMB = sp.symbols('ell bCMB')
b0f = sp.Function('b0')(ell)
rg_eq = sp.Eq(b0f.diff(ell), b0f * (b0f**2 - bCMB**2))
sol = sp.dsolve(rg_eq)
print("RG:", sp.latex(sol))

print("\n=== NumPy/Scipy: Q_D^F Limit ===")
def qdf_num_improved(N_rad=5000, N_theta=200, b0=0.03, H_D=1.0):
    rad = np.logspace(-2, np.log10(10), N_rad)
    theta_grid = np.linspace(0.01, np.pi-0.01, N_theta)
    integ = 0
    dtheta = theta_grid[1] - theta_grid[0]
    for th in theta_grid:
        integrand = b0**2 * np.cos(th)**2 * np.exp(-rad**2/2) / (1 + rad**2)
        integ += simpson(integrand, x=rad) * np.sin(th) * dtheta
    return (2/3) * integ * H_D**2  # Q_D^F = 3.25e-4 → analytic limit


Ns = [100, 1000, 10000]
analytic = (2/3) * 0.03**2
for N in Ns:
    q = qdf_num(N)
    err = abs(q - analytic)/analytic
    print(f"N={N}: Q_D^F={q:.2e}, err={err:.2e}")

print("Ready for FTH Appendix A.1.")



=== SymPy: Full 4D Chern-Ricci via Randers (Pfeifer-Wohlfarth) ===
g_rr: b_{0}^{2} \cos^{2}{\left(\theta \right)} + 1
g_r theta: 0
C^r_theta phi: - \frac{b_{0} y_{2} y_{3} \sin{\left(\theta \right)}}{b_{0}^{2} y_{1}^{2} \cos^{2}{\left(\theta \right)} + r^{2} y_{2}^{2} + r^{2} y_{3}^{2} \sin^{2}{\left(\theta \right)} - y_{0}^{2} + y_{1}^{2}}
R_F approx: \frac{b_{0}^{3}}{r^{2}} - \frac{0.75 b_{0}^{2} y_{2}^{2} y_{3}^{2} \sin^{2}{\left(\theta \right)}}{r^{2} \left(r^{2} y_{2}^{2} + r^{2} y_{3}^{2} \sin^{2}{\left(\theta \right)} - y_{0}^{2} + y_{1}^{2}\right)^{2}}
RG: \left[ b_{0}{\left(\ell \right)} = - bCMB \sqrt{- \frac{1}{e^{bCMB^{2} \left(C_{1} + 2 \ell\right)} - 1}}, \  b_{0}{\left(\ell \right)} = bCMB \sqrt{\frac{1}{1 - e^{bCMB^{2} \left(C_{1} + 2 \ell\right)}}}\right]

=== NumPy/Scipy: Q_D^F Limit ===
N=100: Q_D^F=3.53e-04, err=4.12e-01
N=1000: Q_D^F=3.53e-04, err=4.12e-01
N=10000: Q_D^F=3.53e-04, err=4.12e-01
Ready for FTH Appendix A.1.


In [6]:
import sympy as sp
from scipy.integrate import solve_ivp
import numpy as np

print("=== Python Modul: Linearer Evolutor für Faser-Fluktuationen δy^i(λ) ===")

# SymPy: Linear Matrix A^i_j
lambda_sym = sp.symbols('lambda')
delta_y1, delta_y2, delta_y3 = sp.symbols('delta_y1 delta_y2 delta_y3')
C_r_th_ph_sym = sp.symbols('C_r_theta_phi')
v1, v2, v3 = sp.symbols('v1 v2 v3')

A_matrix = sp.Matrix([[-C_r_th_ph_sym * v3, 0, -C_r_th_ph_sym * v1],
                      [0, 0, 0],
                      [C_r_th_ph_sym * v2, -C_r_th_ph_sym * v1, 0]])
print("A-Matrix:", sp.latex(A_matrix))

# NumPy ODE
def fiber_fluct_ode(t, delta_y, C_params, v):
    """dδy/dλ = A δy mit A^i_j = -C^i_jk v^k"""
    b0, sin_th, F_inv2 = C_params
    C_rthph = -b0 * sin_th * F_inv2
    A = np.array([[-C_rthph * v[2], 0, -C_rthph * v[0]],
                  [0, 0, 0],
                  [C_rthph * v[1], -C_rthph * v[0], 0]])
    return A @ delta_y

# Demo-Integration
C_params = (0.03, np.sin(1.0), 1.0)  # b0, sinθ, 1/F^2
v_demo = np.array([1.0, 0.5, 0.2])
y0 = [1e-3, 5e-4, 2e-4]

sol = solve_ivp(fiber_fluct_ode, [0, 10], y0, args=(C_params, v_demo),
                method='RK45', rtol=1e-8, dense_output=True)

print("δy(λ=10):", sol.y[:,-1])
print("Norm-Wachstum:", np.linalg.norm(sol.y[:,-1]) / np.linalg.norm(y0))
reynolds = np.trace(np.outer(sol.y[:,-1], sol.y[:,-1]))
print("Reynolds ~ ν_eff:", reynolds)

print("Modul ready für FTH Appendix A.2: PDE → ODE → Reynolds-Stress.")


=== Python Modul: Linearer Evolutor für Faser-Fluktuationen δy^i(λ) ===
A-Matrix: \left[\begin{matrix}- C_{r \theta \phi} v_{3} & 0 & - C_{r \theta \phi} v_{1}\\0 & 0 & 0\\C_{r \theta \phi} v_{2} & - C_{r \theta \phi} v_{1} & 0\end{matrix}\right]
δy(λ=10): [0.00110302 0.0005     0.00019354]
Norm-Wachstum: 1.079803712918277
Reynolds ~ ν_eff: 1.5041091153774043e-06
Modul ready für FTH Appendix A.2: PDE → ODE → Reynolds-Stress.
